In [1]:
# Standard Library Imports
import os
import base64
import gzip
import io
import logging
import glob
import time
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
from copy import deepcopy
import multiprocessing as mp
from typing import Mapping, Optional

# Third-Party Imports
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import requests
from matplotlib.backends.backend_pdf import PdfPages
import psycopg2

from astropy.wcs import WCS
from astropy import units as u
from astropy.coordinates import SkyCoord
from astropy.table import Table, vstack, Column
from astropy.cosmology import FlatLambdaCDM
from astropy.io import fits
from astropy.stats import sigma_clipped_stats
from astropy.time import Time
from astropy.visualization import (
    AsymmetricPercentileInterval,
    ImageNormalize,
    LinearStretch,
    LogStretch,
)

from tqdm import tqdm
from requests.adapters import HTTPAdapter
from requests.packages.urllib3.util.retry import Retry
from colorama import init as colorama_init, Fore, Style
import fitsio

from scipy.ndimage import gaussian_filter1d

# Project-Specific Imports
from desispec.io import read_spectra
from desitrip.preproc import rebin_flux, rescale_flux
from desispec.coaddition import coadd_cameras

# Configuration and Constants
# pd.options.mode.copy_on_write = True  # Uncomment if needed
cosmo = FlatLambdaCDM(
    name='Planck18', 
    H0=67.66, 
    Om0=0.30966, 
    Tcmb0=2.7255, 
    Neff=3.046, 
    m_nu=[0., 0., 0.06] * u.eV, 
    Ob0=0.04897
)
cores = 16

In [2]:
TNS_match_DESI = pd.read_csv("DESI_spectra.csv", dtype={
        'TARGETID': str,  # Force string type at read-time
        'NIGHT': str,
        'TILEID': str,
        'PETAL_LOC': str
    })
#TNS_match_DESI["discoverydate"] = pd.to_datetime(TNS_match_DESI["discoverydate"])
#TNS_match_DESI["obsdate"] = pd.to_datetime(TNS_match_DESI["NIGHT"].astype(int).astype(str), format="%Y%m%d")
TNS_match_DESI["NIGHT"] = TNS_match_DESI["NIGHT"].astype(int).astype(str)
TNS_match_DESI["TILEID"] = TNS_match_DESI["TILEID"].astype(int).astype(str)
TNS_match_DESI["PETAL_LOC"] = TNS_match_DESI["PETAL_LOC"].astype(int).astype(str)
#indices = TNS_match_DESI.index[TNS_match_DESI["obsdate"] > TNS_match_DESI["discoverydate"]]
#TNS_match_DESI = TNS_match_DESI.loc[indices]
#TNS_match_DESI['days_since_discovery'] = (TNS_match_DESI['obsdate'] - TNS_match_DESI['discoverydate']).dt.days
#TNS_match_DESI = TNS_match_DESI.sort_values('days_since_discovery', ascending=True)

In [3]:
types = ["Gal", "AGN", "Ia", "Ib", "Ic", "II", "TDE"]
limits = {"Gal": [-np.inf, np.inf],
          "AGN": [-np.inf, np.inf],
          "Ia": [-1, 100],
          "Ib": [-1, 100],
          "Ic": [-1, 100],
          "II": [-1, 100],
          "TDE": [-1, np.inf]}

In [4]:
for current_type in types:
    print("Running on {}".format(current_type))
    current_limit = limits[current_type]
    
    TNS_match_DESI_type = TNS_match_DESI[TNS_match_DESI['TYPE'].str.contains(current_type, case=False, na=False)]

    if(len(TNS_match_DESI_type) > 0):
        TNS_match_DESI_type.to_csv("DESI_ML_training/{}_ToO.csv".format(current_type), index = False)

Running on Gal
Running on AGN
Running on Ia
Running on Ib
Running on Ic
Running on II
Running on TDE


In [5]:
def process_spectrum(i):
    """Processes a single row from the TDE table."""
    path = f'/global/cfs/cdirs/desi/spectro/redux/daily/tiles/cumulative/{i["TILEID"]}/{i["NIGHT"]}/spectra-{i["PETAL_LOC"]}-{i["TILEID"]}-thru{i["NIGHT"]}.fits.gz'
    try:
        spectra = read_spectra(path, targetids=int(i["TARGETID"]))
        iso_time = Time(spectra.fibermap["MJD"], format="mjd").iso[0]
        spectra = coadd_cameras(spectra)
        
        smoothed_flux = gaussian_filter1d(spectra.flux["brz"][0], 5)
        ivar_numeric = np.where(spectra.ivar["brz"] > 0, 1 / spectra.ivar["brz"], np.nan)
        std = np.sqrt(ivar_numeric)
        smoothed_std = gaussian_filter1d(std, 5)
        smoothed_std_clean = np.nan_to_num(smoothed_std, nan=0.0, posinf=0.0, neginf=0.0)
        
        spectrum_table = Table()
        spectrum_table['wavelength'] = spectra.wave["brz"]
        spectrum_table['flux'] = smoothed_flux
        spectrum_table['fluxerr'] = smoothed_std_clean[0]
        
        spectrum_table.to_pandas().to_csv(f"DESI_ML_training/spectra/{int(i['TARGETID'])}.csv")
        
        fig = plt.figure(figsize=[20,12])
        plt.plot(spectrum_table['wavelength'], spectrum_table['flux'])
        plt.title(f"TARGETID: {int(i['TARGETID'])} TYPE: {i['type']} Z: {i['Z']} PHASE: {i['days_since_discovery']}")
        plt.xlabel("Angstroms")
        plt.ylabel("Relative Flux")
        plt.show()
        
        return fig
    except Exception as e:
        print(f"Error processing TARGETID {i['TARGETID']}: {e}")
        return None

In [6]:
def save_plots_to_pdf(figures, filename):
    with PdfPages(filename) as p:
        for fig in figures:
            if fig is not None:
                p.savefig(fig)
                plt.close(fig)

In [7]:
if __name__ == "__main__":
    for current_type in types:
        print("Running on {}".format(current_type))
        current_limit = limits[current_type]
        
        TNS_match_DESI_type = TNS_match_DESI[TNS_match_DESI['TYPE'].str.contains(current_type, case=False, na=False)]
        
        with mp.Pool(cores) as pool:
            results = list(tqdm(pool.imap(process_spectrum, (row[1] for row in TNS_match_DESI_type.iterrows())), total=len(TNS_match_DESI_type)))
        save_plots_to_pdf(results, filename = "DESI_ML_training/{}_ToO.pdf".format(current_type))

Running on Gal


  0%|          | 0/1 [00:00<?, ?it/s]

INFO:spectra.py:451:read_spectra: iotime 2.134 sec to read spectra from:  spectra-9-26527-thru20250206.fits.gz at 2025-06-12T00:48:28.401783


/tmp/ipykernel_880703/2385020135.py:10: RuntimeWarning: divide by zero encountered in true_divide
  ivar_numeric = np.where(spectra.ivar["brz"] > 0, 1 / spectra.ivar["brz"], np.nan)


Error processing TARGETID 43977655305200184: 'type'


...


  0%|          | 0/1 [00:00<?, ?it/s]

INFO:spectra.py:451:read_spectra: iotime 2.060 sec to read spectra from:  spectra-2-2327-thru20250105.fits.gz at 2025-06-12T00:48:40.420241


/tmp/ipykernel_880703/2385020135.py:10: RuntimeWarning: divide by zero encountered in true_divide
  ivar_numeric = np.where(spectra.ivar["brz"] > 0, 1 / spectra.ivar["brz"], np.nan)


Error processing TARGETID 43978677352226198: 'type'


100%|██████████| 1/1 [00:02<00:00,  2.15s/it]
